# Notebook 04 — Fund Performance Analytics
**Bluestock MF Capstone — D4**

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats
from pathlib import Path

PROC = Path('..') / 'data' / 'processed'
RF   = 0.065
DAYS = 252


## 1. Daily Returns

In [ ]:

nav = pd.read_csv(PROC/'02_nav_history_clean.csv', parse_dates=['date'])
nav = nav.sort_values(['amfi_code','date'])
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
nav.to_csv(PROC/'11_nav_with_returns.csv', index=False)
print(nav.groupby('amfi_code')['daily_return'].describe().head())


## 2. CAGR Table — 1yr, 3yr, 5yr

In [ ]:

def cagr(group, years):
    end = group.iloc[-1]; start_d = end['date'] - pd.DateOffset(years=years)
    period = group[group['date'] >= start_d]
    if len(period) < 10: return np.nan
    n = (period['date'].iloc[-1] - period['date'].iloc[0]).days / 365.25
    return (period['nav'].iloc[-1] / period['nav'].iloc[0]) ** (1/n) - 1 if n > 0 else np.nan

cagr_rows = []
for code, grp in nav.groupby('amfi_code'):
    cagr_rows.append({'amfi_code': code,
                      'cagr_1yr': cagr(grp,1), 'cagr_3yr': cagr(grp,3), 'cagr_5yr': cagr(grp,5)})
cagr_df = pd.DataFrame(cagr_rows)
funds = pd.read_csv(PROC/'01_fund_master_clean.csv')
cagr_df = cagr_df.merge(funds[['amfi_code','scheme_name','category']], on='amfi_code')
print(cagr_df.sort_values('cagr_3yr', ascending=False).head(10).to_string(index=False))


## 3. Sharpe & Sortino Ratios

In [ ]:

def sharpe(r):
    r = r.dropna(); rf = RF/DAYS
    return ((r - rf).mean() / r.std() * np.sqrt(DAYS)) if r.std() > 0 else np.nan

def sortino(r):
    r = r.dropna(); rf = RF/DAYS
    ds = r[r < 0].std()
    return ((r - rf).mean() / ds * np.sqrt(DAYS)) if ds > 0 else np.nan

ratios = nav.groupby('amfi_code')['daily_return'].agg(sharpe=sharpe, sortino=sortino).reset_index()
ratios = ratios.merge(funds[['amfi_code','scheme_name']], on='amfi_code')
print('Top 10 by Sharpe:')
print(ratios.sort_values('sharpe', ascending=False).head(10)[['scheme_name','sharpe','sortino']].to_string(index=False))


## 4. Alpha & Beta via OLS

In [ ]:

bench = pd.read_csv(PROC/'10_benchmark_indices_clean.csv', parse_dates=['date'])
nifty = bench[bench['index_name']=='NIFTY100'].set_index('date')['close_value'].pct_change()

ab_rows = []
for code, grp in nav.groupby('amfi_code'):
    r = grp.set_index('date')['daily_return']
    merged = pd.concat([r, nifty.rename('bench')], axis=1).dropna()
    if len(merged) < 30: continue
    slope, intercept, *_ = stats.linregress(merged['bench'], merged[r.name if r.name else 'daily_return'])
    ab_rows.append({'amfi_code': code, 'alpha': intercept*DAYS, 'beta': slope})
ab_df = pd.DataFrame(ab_rows).merge(funds[['amfi_code','scheme_name']], on='amfi_code')
ab_df.to_csv(PROC/'alpha_beta.csv', index=False)
print(ab_df.sort_values('alpha', ascending=False).head(10).to_string(index=False))


## 5. Maximum Drawdown

In [ ]:

dd_rows = []
for code, grp in nav.groupby('amfi_code'):
    n = grp['nav']
    mdd = (n / n.cummax() - 1).min()
    dd_rows.append({'amfi_code': code, 'max_drawdown': mdd})
dd_df = pd.DataFrame(dd_rows).merge(funds[['amfi_code','scheme_name','category']], on='amfi_code')
print('Worst drawdowns:')
print(dd_df.sort_values('max_drawdown').head(10).to_string(index=False))


## 6. Fund Scorecard (0–100)

In [ ]:

scorecard = pd.read_csv(PROC/'fund_scorecard.csv')
print(scorecard[['scheme_name','cagr_3yr','sharpe','alpha','scorecard']].head(15).to_string(index=False))

# Chart
fig, ax = plt.subplots(figsize=(10,6))
top = scorecard.head(15)
ax.barh(top['scheme_name'].str[:35], top['scorecard'], color='steelblue')
ax.set_xlabel('Composite Scorecard (0–100)')
ax.set_title('Top 15 Funds by Composite Scorecard')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/fund_scorecard.png', dpi=150)
plt.show()


## 7. Benchmark Comparison — Top 5 vs Nifty 50

In [ ]:

top5_codes = scorecard.head(5)['amfi_code'].tolist()
top5_nav = nav[nav['amfi_code'].isin(top5_codes)].copy()
top5_nav = top5_nav.merge(funds[['amfi_code','scheme_name']], on='amfi_code')

# Normalise to 100
nifty50 = bench[bench['index_name']=='NIFTY50'].copy()
nifty50['scheme_name'] = 'NIFTY 50'
nifty50 = nifty50.rename(columns={'close_value':'nav','index_name':'amfi_code'})[['date','nav','scheme_name']]

fig = px.line(top5_nav, x='date', y='nav', color='scheme_name',
              title='Top 5 Funds vs NIFTY50 (raw NAV, 2022-2025)')
fig.show()
print('Tracking error computed separately in compute_metrics.py')
